# Amyloid Core Prediction - ESM2 Ensemble Training

This notebook replicates `scripts/train_ensemble_esm2.py` for Google Colab (VSCode direct connection).

**Pipeline summary:**
```
sequences  ->  ESM2 (per-residue -> pool -> PCA)  ->  concatenate with optional
               builtin (540-dim) / aaindex (304-dim)  ->  6 classifiers trained
               with k-fold CV + random hyperparameter search  ->  3 ensembles saved
```

**Before running:**
1. Enable GPU: *Runtime -> Change runtime type -> T4 GPU*
2. Edit the **Configuration** cell (Cell 3) - that is the only cell you need to change

## 1 - Environment

In [ ]:
import os, subprocess, sys

result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                         '--format=csv,noheader'], capture_output=True, text=True)
if result.returncode == 0:
    print('GPU:', result.stdout.strip())
else:
    print('No GPU detected - ESM2 inference will be slow on CPU.')
    print('Enable GPU via Runtime -> Change runtime type -> T4 GPU')

print('Python:', sys.version.split()[0])

In [ ]:
!pip install -q --upgrade transformers accelerate tqdm pandas tensorboard
print('Dependencies ready.')

## 2 - Setup

Clones the repo from GitHub into `/content/` (or pulls if already present).
No Drive mounting needed when using VSCode direct Colab connection.

In [ ]:
import os, sys
from pathlib import Path

REPO_URL  = 'https://github.com/NGUYENTHAISON-2311/multi_model_pipeline_Son'
REPO_NAME = 'multi_model_pipeline_Son'
PIPELINE_DIR = f'/content/{REPO_NAME}'

pipeline_root = Path(PIPELINE_DIR)

if not pipeline_root.exists():
    print(f'Cloning {REPO_URL} ...')
    !git clone {REPO_URL} {PIPELINE_DIR}
else:
    print(f'Repo already exists at {PIPELINE_DIR} — pulling latest changes ...')
    !git -C {PIPELINE_DIR} pull

scripts_dir = pipeline_root / 'scripts'
for p in [str(pipeline_root), str(scripts_dir)]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(str(pipeline_root))
print('Working directory:', os.getcwd())

## 3 - Configuration

**Edit this cell** - all run parameters are here.

In [ ]:
from pathlib import Path

# Data paths (relative to PIPELINE_DIR or absolute)
POSITIVE_JSON = 'cores_07/train_core_set_seed_20.json'
NEGATIVE_JSON = 'cores_07/len_matching_filtered_disordered_regions_clustered.json'

# Output / cache (local paths)
OUTPUT_DIR = 'outputs/training'
CACHE_DIR  = 'outputs/cache'   # set None to disable caching

# Run label
TITLE = 'Colab ESM2 run'

# Feature tokens (order = concatenation order)
#   'builtin'  - 540-dim handcrafted vector (requires IUPred3)
#   'aaindex'  - 304-dim AAindex mean features
#   'esm2'     - ESM2 embeddings
#   '<path>'   - path to a custom JSON lookup-table spec
FEATURES = ['esm2']            # e.g. ['builtin', 'esm2'] or ['aaindex', 'esm2']

# ESM2 settings
ESM2_MODEL = 'facebook/esm2_t12_35M_UR50D'   # 35M - fast; use t33_650M for best quality
ESM2_POOL  = 'mean'                           # 'mean' | 'max' | 'cls'
ESM2_DIM   = 64                               # PCA target dims; 0 = no PCA (keep full)

# Pre-computed embedding files (skip ESM2 inference if provided)
EMBEDDINGS_POS = None   # e.g. 'outputs/cache/pos_emb.pt'
EMBEDDINGS_NEG = None

# Pre-computed IUPred scores (only needed when 'builtin' is in FEATURES)
IUPRED_SCORES_POS = None
IUPRED_SCORES_NEG = None

# Training
N_COMBOS  = 10          # random hyperparameter combos per algorithm
N_FOLDS   = 5           # k-fold CV splits
METRIC    = 'F1_score'  # combo selection metric: F1_score | MCC | Accuracy | ...
N_WORKERS = 0           # combo threads per algorithm; 0 = all CPUs

# Restrict to a subset of algorithms (None = all six)
ALGORITHMS = None       # e.g. ['random_forest', 'extra_trees', 'svm']

TENSORBOARD = True

print('Configuration loaded.')
print(f'  Features : {FEATURES}')
print(f'  ESM2     : {ESM2_MODEL}  pool={ESM2_POOL}  pca_dim={ESM2_DIM}')
print(f'  Training : {N_COMBOS} combos x {N_FOLDS} folds  metric={METRIC}  workers={N_WORKERS}')

## 4 - Load data

In [ ]:
import json
from pathlib import Path

from src.configuration import load_config, resolve_pipeline_path
from src.training_pipeline import SUPPORTED_MODELS, run_ensemble_training

config = load_config(None)
training_cfg = config['training']

def _load_sequences(path):
    path = Path(path)
    if path.suffix.lower() == '.json':
        with path.open() as f:
            records = json.load(f)
        seqs, ids = [], []
        for i, r in enumerate(records):
            seq = r.get('Sequence') or r.get('sequence')
            if seq is None:
                raise KeyError(f'Record {i} has no Sequence field')
            seqs.append(seq)
            ids.append(r.get('ID') or r.get('core_id') or str(i))
    else:
        from src.feature_pipeline import load_sequences_from_file
        seqs = load_sequences_from_file(path)
        ids  = [str(i) for i in range(len(seqs))]
    return ids, seqs

pos_ids, pos_seqs = _load_sequences(POSITIVE_JSON)
neg_ids, neg_seqs = _load_sequences(NEGATIVE_JSON)

print(f'Positive sequences : {len(pos_seqs)}')
print(f'Negative sequences : {len(neg_seqs)}')
print(f'Example positive   : {pos_seqs[0][:60]} ...')
print(f'Example negative   : {neg_seqs[0][:60]} ...')

## 5 - Build feature matrices

Cells 5a-5c build the feature matrix.
If `CACHE_DIR` is set and a cached matrix exists, it is reloaded and all build cells are skipped automatically.

In [ ]:
import numpy as np
import pickle

cache_dir   = Path(CACHE_DIR) if CACHE_DIR else None
_cache_pos  = cache_dir / 'pos_matrix.npy'    if cache_dir else None
_cache_neg  = cache_dir / 'neg_matrix.npy'    if cache_dir else None
_cache_pca  = cache_dir / 'pca_reducer.pkl'   if cache_dir else None
_cache_info = cache_dir / 'feature_info.json' if cache_dir else None

if cache_dir and _cache_pos.exists() and _cache_neg.exists():
    print(f'[Cache] Loading feature matrices from {cache_dir}')
    pos_matrix   = np.load(_cache_pos)
    neg_matrix   = np.load(_cache_neg)
    feature_info = json.loads(_cache_info.read_text()) if _cache_info.exists() else {}
    pca_reducer  = None
    if _cache_pca and _cache_pca.exists():
        with _cache_pca.open('rb') as fh:
            pca_reducer = pickle.load(fh).get('pca')
    dim_label   = feature_info.get('dim_label', f'{pos_matrix.shape[1]}-dim (from cache)')
    _skip_build = True
    print(f'  pos {pos_matrix.shape}  neg {neg_matrix.shape}')
    print(f'  {dim_label}')
else:
    _skip_build = False
    parts_pos, parts_neg, dim_parts = [], [], []
    feature_info = {}
    pca_reducer  = None
    print('No cache found - building feature matrices from scratch.')

In [ ]:
# 5a: Pipeline features (builtin / aaindex / custom JSON)

AAINDEX_SPEC = 'data/aaindex_features.json'

pipeline_specs = []
use_esm2 = 'esm2' in FEATURES
for tok in FEATURES:
    if tok == 'esm2':      continue
    elif tok == 'aaindex': pipeline_specs.append(AAINDEX_SPEC)
    else:                  pipeline_specs.append(tok)

if not _skip_build and pipeline_specs:
    from src.feature_loader import load_and_prepare_feature_specs, compute_sequence_feature_matrix
    from src.feature_pipeline import compute_average_iupred_scores_from_sequences

    prepared    = load_and_prepare_feature_specs(config, pipeline_specs)
    has_builtin = any(s.get('_builtin') for s in prepared)

    def _iupred(seqs, cached_path):
        if cached_path:
            with open(cached_path) as fh:
                raw = json.load(fh)
            return [float(raw[i]) if isinstance(raw, dict) else float(raw[i])
                    for i in range(len(seqs))]
        iupred_script = resolve_pipeline_path(config, training_cfg['iupred_script'])
        itype = training_cfg.get('iupred_input_type', 'long')
        print(f'  Running IUPred ({itype}) on {len(seqs)} sequences ...')
        return compute_average_iupred_scores_from_sequences(seqs, iupred_script, itype)

    print(f'[1/2] Pipeline features: {pipeline_specs}')
    pos_iupred = _iupred(pos_seqs, IUPRED_SCORES_POS) if has_builtin else [0.0]*len(pos_seqs)
    neg_iupred = _iupred(neg_seqs, IUPRED_SCORES_NEG) if has_builtin else [0.0]*len(neg_seqs)

    pf_pos = np.array(compute_sequence_feature_matrix(pos_seqs, prepared, pos_iupred,
                                                       desc='  Positives'), dtype=np.float32)
    pf_neg = np.array(compute_sequence_feature_matrix(neg_seqs, prepared, neg_iupred,
                                                       desc='  Negatives'), dtype=np.float32)
    parts_pos.append(pf_pos)
    parts_neg.append(pf_neg)
    dim_parts.append(f'pipeline:{pf_pos.shape[1]}')
    feature_info['pipeline_dim'] = int(pf_pos.shape[1])
    print(f'  pos {pf_pos.shape}  neg {pf_neg.shape}')
elif _skip_build:
    print('Skipped (loaded from cache).')
else:
    print('No pipeline features requested.')

In [ ]:
# 5b: ESM2 embeddings -> pool -> optional PCA

if not _skip_build and use_esm2:
    import torch
    from esm2_embedding import load_model as _load_esm2, embed_sequences

    step = '2/2' if pipeline_specs else '1/1'
    print(f'[{step}] ESM2  model={ESM2_MODEL}  pool={ESM2_POOL}')

    def _pool_hidden(hidden_states, attention_mask, strategy):
        if strategy == 'cls':
            return hidden_states[:, 0, :]
        if strategy == 'max':
            mask = attention_mask.unsqueeze(-1).bool()
            return hidden_states.masked_fill(~mask, float('-inf')).max(dim=1).values
        mask = attention_mask.unsqueeze(-1).float()
        return (hidden_states * mask).sum(dim=1) / mask.sum(dim=1)

    def _embed(seqs, pt_path):
        if pt_path:
            print(f'  Loading pre-computed from {pt_path}')
            from feature_builder import _load_embeddings_from_pt
            return _load_embeddings_from_pt(pt_path, [str(i) for i in range(len(seqs))])
        tokenizer, model, device = _load_esm2(ESM2_MODEL)
        print(f'  device={device}  hidden_size={model.config.hidden_size}')
        batch_size = 16
        all_pooled = []
        for i in range(0, len(seqs), batch_size):
            batch  = seqs[i : i + batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                outputs = model(**inputs)
            pooled = _pool_hidden(outputs.last_hidden_state, inputs['attention_mask'], ESM2_POOL)
            all_pooled.append(pooled.cpu())
            if (i // batch_size) % 5 == 0:
                print(f'  {i + len(batch)}/{len(seqs)} sequences embedded ...')
        return torch.cat(all_pooled, dim=0).numpy().astype(np.float32)

    print('  Embedding positives ...')
    esm_pos = _embed(pos_seqs, EMBEDDINGS_POS)
    print('  Embedding negatives ...')
    esm_neg = _embed(neg_seqs, EMBEDDINGS_NEG)
    print(f'  Raw embedding  pos {esm_pos.shape}  neg {esm_neg.shape}')
    feature_info['esm2_raw_dim'] = int(esm_pos.shape[1])

    if ESM2_DIM and ESM2_DIM > 0:
        from sklearn.decomposition import PCA
        n_comp = min(ESM2_DIM, esm_pos.shape[1], esm_pos.shape[0] + esm_neg.shape[0])
        print(f'  PCA: {esm_pos.shape[1]} -> {n_comp} dims (fitted on combined pos+neg)')
        combined    = np.concatenate([esm_pos, esm_neg], axis=0)
        pca_reducer = PCA(n_components=n_comp, random_state=42)
        pca_reducer.fit(combined)
        var = pca_reducer.explained_variance_ratio_.sum()
        print(f'  Explained variance: {var:.3f}')
        esm_pos = pca_reducer.transform(esm_pos).astype(np.float32)
        esm_neg = pca_reducer.transform(esm_neg).astype(np.float32)
        dim_parts.append(f'esm2-pca:{esm_pos.shape[1]}')
    else:
        dim_parts.append(f'esm2:{esm_pos.shape[1]}')

    parts_pos.append(esm_pos)
    parts_neg.append(esm_neg)
    print(f'  After reduction  pos {esm_pos.shape}  neg {esm_neg.shape}')
elif _skip_build:
    print('Skipped (loaded from cache).')
else:
    print('ESM2 not in FEATURES - skipped.')

In [ ]:
# 5c: Concatenate parts and save to cache

if not _skip_build:
    pos_matrix = np.concatenate(parts_pos, axis=1)
    neg_matrix = np.concatenate(parts_neg, axis=1)
    dim_label  = ' + '.join(dim_parts) + f'  ->  {pos_matrix.shape[1]} total dims'
    feature_info['total_dim'] = int(pos_matrix.shape[1])
    feature_info['dim_label'] = dim_label
    print(f'Final feature matrix: pos {pos_matrix.shape}  neg {neg_matrix.shape}')
    print(f'Feature breakdown   : {dim_label}')

    if cache_dir is not None:
        cache_dir.mkdir(parents=True, exist_ok=True)
        np.save(_cache_pos, pos_matrix)
        np.save(_cache_neg, neg_matrix)
        _cache_info.write_text(json.dumps(feature_info, indent=2))
        if pca_reducer is not None:
            with _cache_pca.open('wb') as fh:
                pickle.dump({'pca': pca_reducer, 'esm2_model': ESM2_MODEL,
                             'esm2_pool': ESM2_POOL}, fh)
        print(f'[Cache] Saved -> {cache_dir}')
else:
    dim_label = feature_info.get('dim_label', f'{pos_matrix.shape[1]}-dim')

print(f'\nReady to train: {pos_matrix.shape[1]}-dim features, '
      f'{pos_matrix.shape[0]} pos + {neg_matrix.shape[0]} neg samples.')

## 6 - Train

In [ ]:
from src.training_pipeline import run_ensemble_training, SUPPORTED_MODELS

if ALGORITHMS:
    valid = {a.lower() for a in ALGORITHMS}
    all_algos = config['training'].get('algorithms', [])
    config['training']['algorithms'] = [a for a in all_algos if a['type'] in valid]
    print(f'Restricted to: {valid}')

if TITLE:
    bar = '=' * max(60, len(TITLE) + 4)
    print(f'\n{bar}\n  Run: {TITLE}\n{bar}\n')

ensembles, artifacts = run_ensemble_training(
    config,
    positive_path=POSITIVE_JSON,
    negative_path=NEGATIVE_JSON,
    output_dir=OUTPUT_DIR,
    n_combos=N_COMBOS,
    n_folds=N_FOLDS,
    optimization_metric=METRIC,
    n_workers=N_WORKERS,
    positive_features_matrix=pos_matrix,
    negative_features_matrix=neg_matrix,
    feature_dim_label=dim_label,
)

## 7 - Save ESM2 PCA reducer

In [ ]:
run_dir = Path(artifacts['run_dir'])
run_id  = run_dir.name

if pca_reducer is not None:
    pca_path = run_dir / 'esm2_pca_reducer.pkl'
    with open(pca_path, 'wb') as fh:
        pickle.dump({
            'pca':           pca_reducer,
            'esm2_model':    ESM2_MODEL,
            'esm2_pool':     ESM2_POOL,
            'n_components':  int(pca_reducer.n_components_),
            'var_explained': float(pca_reducer.explained_variance_ratio_.sum()),
        }, fh)
    print(f'ESM2 PCA reducer -> {pca_path}')

lines = [f'  Run ID : {run_id}']
if TITLE:
    lines.insert(0, f'  Title  : {TITLE}')
bar = '=' * max(60, max(len(l) for l in lines) + 4)
print(f'\n{bar}')
print(f'  Training completed - {len(ensembles)} ensemble variant(s)')
for l in lines:
    print(l)
print(bar)
print(f"\nRun directory        : {artifacts['run_dir']}")
print(f"soft_ensemble.pkl    : {artifacts['soft_pkl']}")
print(f"weighted_ensemble.pkl: {artifacts['weighted_pkl']}")
print(f"best_model.pkl       : {artifacts['best_pkl']}")
print(f"Summary CSV          : {artifacts['summary_csv']}")

## 8 - Results summary

In [ ]:
import pandas as pd

summary_csv = artifacts.get('summary_csv')
if summary_csv and Path(summary_csv).exists():
    df = pd.read_csv(summary_csv)
    display(df.set_index('algorithm').style
              .format(precision=4)
              .highlight_max(subset=[c for c in df.columns if 'mean' in c.lower()],
                             color='#d4edda')
              .set_caption(f'Training summary - {run_id}'))
else:
    print('summary.csv not found.')

In [ ]:
import matplotlib.pyplot as plt
import csv as _csv

per_algo_dir = run_dir / 'per_algorithm'
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
colors = plt.cm.tab10.colors

for ci, algo_dir in enumerate(sorted(per_algo_dir.iterdir())):
    scores_csv = algo_dir / 'scores.csv'
    if not scores_csv.exists():
        continue
    with scores_csv.open(newline='') as fh:
        rows = list(_csv.DictReader(fh))
    folds = [int(r['Fold'])       for r in rows]
    f1s   = [float(r['F1_score']) for r in rows]
    mccs  = [float(r['MCC'])      for r in rows]
    c = colors[ci % len(colors)]
    axes[0].plot(folds, f1s,  marker='o', label=algo_dir.name, color=c)
    axes[1].plot(folds, mccs, marker='o', label=algo_dir.name, color=c)

for ax, title in zip(axes, ['F1 score per fold', 'MCC per fold']):
    ax.set_xlabel('Fold'); ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle(f'{TITLE} - {run_id}', fontsize=11)
plt.tight_layout()
plt.savefig(str(run_dir / 'fold_metrics.png'), dpi=120)
plt.show()

## 9 - TensorBoard (optional)

Logs are written to `<run_dir>/tensorboard/`.
View in the VSCode terminal:
```
tensorboard --logdir outputs/training/run_YYYYMMDD_HHMMSS/tensorboard
```

In [ ]:
if TENSORBOARD:
    try:
        from torch.utils.tensorboard import SummaryWriter
    except ImportError:
        SummaryWriter = None

    if SummaryWriter:
        tb_dir = run_dir / 'tensorboard'
        writer = SummaryWriter(log_dir=str(tb_dir))

        if TITLE:
            writer.add_text('run/title', TITLE, 0)
        writer.add_scalar('features/total_dim', feature_info.get('total_dim', 0), 0)
        if feature_info.get('pipeline_dim'):
            writer.add_scalar('features/pipeline_dim', feature_info['pipeline_dim'], 0)
        if feature_info.get('esm2_raw_dim'):
            writer.add_scalar('features/esm2_raw_dim', feature_info['esm2_raw_dim'], 0)
        if pca_reducer is not None:
            writer.add_scalar('features/esm2_pca_explained_variance',
                              float(pca_reducer.explained_variance_ratio_.sum()), 0)
            writer.add_scalar('features/esm2_pca_components', int(pca_reducer.n_components_), 0)
        writer.add_text('features/config',
                        f'esm2_model={ESM2_MODEL}  pool={ESM2_POOL}  pca_dim={ESM2_DIM}', 0)

        metrics = ['F1_score', 'Accuracy', 'Precision', 'Recall', 'MCC']
        for algo_dir in sorted((run_dir / 'per_algorithm').iterdir()):
            if not algo_dir.is_dir(): continue
            algo = algo_dir.name
            scores_csv = algo_dir / 'scores.csv'
            if scores_csv.exists():
                with scores_csv.open(newline='') as fh:
                    for row in _csv.DictReader(fh):
                        fold = int(row['Fold']) - 1
                        for m in metrics:
                            writer.add_scalar(f'{algo}/fold/{m}', float(row[m]), fold)
            summary_json = algo_dir / 'summary.json'
            if summary_json.exists():
                with summary_json.open() as fh:
                    s = json.load(fh)
                for m in metrics:
                    writer.add_scalar(f'{algo}/summary/mean_{m}', s[f'mean_{m}'], 0)
                    writer.add_scalar(f'{algo}/summary/std_{m}',  s[f'std_{m}'],  0)

        writer.close()
        print(f'TensorBoard logs -> {tb_dir}')
        print(f'Run: tensorboard --logdir "{tb_dir}"')
    else:
        print('TensorBoard not available. Install with: pip install tensorboard')
else:
    print('TensorBoard disabled (set TENSORBOARD = True to enable).')